In [1]:

# STUDENT HEALTH & WELLNESS PORTAL

# What this program does:
#   1. Lets a user log in
#   2. Lets you add a wellness record (sleep, mood, exercise)
#   3. Lets you search records by student name
#   4. Shows a simple report
#   5. Saves everything in a file so it is not lost

# Login for testing:  username = admin   password = 1234


import json   # json helps us save and load data in a file

# The names of the files where we keep our data
USERS_FILE = "users.json"
RECORDS_FILE = "records.json"



# PART 1: SAVING AND LOADING (File Handling)


def load_file(filename, empty_value):
    """Opens a file and returns what is inside it.
    If the file does not exist, it returns empty_value instead."""
    try:
        file = open(filename, "r")
        data = json.load(file)
        file.close()
        return data
    except FileNotFoundError:
        # First time running the program: no file yet, that's okay!
        return empty_value
    except PermissionError:
        print("Sorry, I am not allowed to open", filename)
        return empty_value
    except json.JSONDecodeError:
        print("The file", filename, "is damaged. Starting fresh.")
        return empty_value


def save_file(filename, data):
    """Saves data into a file. Returns True if it worked, False if not."""
    try:
        file = open(filename, "w")
        json.dump(data, file, indent=4)
        file.close()
        return True
    except PermissionError:
        print("Sorry, I am not allowed to save to", filename)
        return False



# PART 2: LOGIN

def user_login(users):
    """Asks for username and password.
    Returns the username if correct, or None if login fails."""
    username = input("Username: ").strip().lower()
    password = input("Password: ").strip()

    # Check if this username exists AND the password matches
    if username in users and users[username] == password:
        print("Login successful! Welcome,", username)
        return username
    else:
        print("Wrong username or password.")
        return None


def register_user(users):
    """Creates a new account and saves it."""
    username = input("Choose a username: ").strip().lower()

    if username == "":
        print("Username cannot be empty.")
    elif username in users:
        print("That username is already taken.")
    else:
        password = input("Choose a password: ").strip()
        users[username] = password
        save_file(USERS_FILE, users)
        print("Account created! You can now log in.")



# PART 3: ADDING A RECORD


def ask_for_number(question):
    """Keeps asking until the user types a valid number.
    Returns that number."""
    while True:
        try:
            number = float(input(question))
            return number
        except ValueError:
            # This runs if the user types letters like "abc"
            print("Oops! Please type a number, like 7 or 7.5")


def add_record(records, username):
    """Asks the user for their wellness details and saves them.
    Returns the new record."""
    print("\n--- Add a New Wellness Record ---")

    name = input("Student name: ").strip()
    date = input("Date (example 2026-11-13): ").strip()
    sleep = ask_for_number("Hours of sleep last night: ")
    mood = ask_for_number("Mood from 1 (sad) to 10 (happy): ")
    exercise = ask_for_number("Minutes of exercise today: ")

    # Make sure mood is between 1 and 10
    while mood < 1 or mood > 10:
        print("Mood must be between 1 and 10.")
        mood = ask_for_number("Mood from 1 (sad) to 10 (happy): ")

   
    new_record = {
        "name": name,
        "date": date,
        "sleep": sleep,
        "mood": mood,
        "exercise": exercise,
        "added_by": username
    }

    records.append(new_record)          # put it in our list
    save_file(RECORDS_FILE, records)    # save the list to the file
    print("Record saved!")

    # Give a little friendly advice
    if sleep < 6:
        print("Tip: You slept less than 6 hours. Try to rest more!")
    if mood <= 4:
        print("Tip: Feeling low? Talking to someone can really help.")

    return new_record



# PART 4: SEARCHING AND SHOWING RECORDS


def search_record(records, search_name):
    """Looks through all records and returns the ones
    that match the name typed by the user."""
    found = []
    for record in records:
        
        if search_name.lower() in record["name"].lower():
            found.append(record)
    return found


def show_records(records):
    """Prints a list of records on the screen."""
    if len(records) == 0:
        print("No records found.")
        return

    for record in records:
        print("-----------------------------")
        print("Name    :", record["name"])
        print("Date    :", record["date"])
        print("Sleep   :", record["sleep"], "hours")
        print("Mood    :", record["mood"], "/ 10")
        print("Exercise:", record["exercise"], "minutes")
    print("-----------------------------")
    print("Total records shown:", len(records))



# PART 5: REPORT


def generate_report(records):
    """Works out averages and prints a simple report.
    Returns the report as text."""
    if len(records) == 0:
        print("There is no data yet, so no report.")
        return ""

    total_sleep = 0
    total_mood = 0
    total_exercise = 0
    unhappy_students = []

   
    for record in records:
        total_sleep = total_sleep + record["sleep"]
        total_mood = total_mood + record["mood"]
        total_exercise = total_exercise + record["exercise"]

        if record["mood"] <= 4:
            unhappy_students.append(record["name"])

    count = len(records)
    report = "\n===== WELLNESS REPORT =====\n"
    report = report + "Total records   : " + str(count) + "\n"
    report = report + "Average sleep   : " + str(round(total_sleep / count, 1)) + " hours\n"
    report = report + "Average mood    : " + str(round(total_mood / count, 1)) + " / 10\n"
    report = report + "Average exercise: " + str(round(total_exercise / count, 1)) + " minutes\n"

    if len(unhappy_students) > 0:
        report = report + "Students with low mood: " + ", ".join(unhappy_students) + "\n"
    else:
        report = report + "No students with low mood. Great!\n"

    print(report)

    # Also save the report to a text file
    try:
        file = open("report.txt", "w")
        file.write(report)
        file.close()
        print("(Report also saved in report.txt)")
    except PermissionError:
        print("Could not save the report file.")

    return report



# PART 6: MENUS (the main loop that keeps things running)


def portal_menu(username, records):
    """The menu shown after login. Repeats until user logs out."""
    while True:
        print("\n===== MAIN MENU =====")
        print("1. Add a record")
        print("2. Search records")
        print("3. See all records")
        print("4. Generate report")
        print("5. Log out")

        choice = input("Pick a number: ").strip()

        if choice == "1":
            add_record(records, username)
        elif choice == "2":
            name = input("Type a name to search: ")
            results = search_record(records, name)
            show_records(results)
        elif choice == "3":
            show_records(records)
        elif choice == "4":
            generate_report(records)
        elif choice == "5":
            print("Logged out. See you soon!")
            break    
        else:
            print("That is not a valid choice. Try 1 to 5.")


def main():
    """This is where the program starts."""
    
    users = load_file(USERS_FILE, {})
    records = load_file(RECORDS_FILE, [])

    
    if "admin" not in users:
        users["admin"] = "1234"
        save_file(USERS_FILE, users)

    print("Welcome to the Student Health & Wellness Portal!")

    
    while True:
        print("\n===== START MENU =====")
        print("1. Login")
        print("2. Create account")
        print("3. Exit")

        choice = input("Pick a number: ").strip()

        if choice == "1":
            username = user_login(users)
            if username is not None:
                portal_menu(username, records)
        elif choice == "2":
            register_user(users)
        elif choice == "3":
            print("Goodbye! Take care of yourself!")
            break
        else:
            print("That is not a valid choice. Try 1, 2 or 3.")



main()

Welcome to the Student Health & Wellness Portal!

===== START MENU =====
1. Login
2. Create account
3. Exit


Pick a number:  1
Username:  admin
Password:  1234


Login successful! Welcome, admin

===== MAIN MENU =====
1. Add a record
2. Search records
3. See all records
4. Generate report
5. Log out


Pick a number:  4


There is no data yet, so no report.

===== MAIN MENU =====
1. Add a record
2. Search records
3. See all records
4. Generate report
5. Log out


Pick a number:  4


... (15 outputs left out)

Pick a number:  5


Logged out. See you soon!

===== START MENU =====
1. Login
2. Create account
3. Exit


Pick a number:  3


Goodbye! Take care of yourself!
